In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [61]:
df = pd.read_csv("diabetes.csv")

In [62]:
df.shape

(768, 9)

In [63]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [64]:
X = df.drop("Outcome" , axis=1)
y = df["Outcome"]

In [65]:
df["Outcome"].value_counts(normalize=True)

Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64

In [66]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.3 , random_state=42 , stratify=y)

In [67]:
from sklearn.preprocessing import StandardScaler

std = StandardScaler()

X_train = std.fit_transform(X_train)
X_test = std.transform(X_test)

In [68]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , Sequential

In [69]:
model = Sequential([
    layers.Dense(5 , activation="relu" , input_shape=(X_train.shape[1],) ),
    layers.Dense(3 , activation="relu"),
    layers.Dense(1 , activation="sigmoid")
])
model.summary()

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67 (268.00 B)

 Trainable params: 67 (268.00 B)

 Non-trainable params: 0 (0.00 B)

In [70]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate = 0.01),
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [71]:
model.fit(
    X_train , y_train , 
    batch_size = 32 , 
    epochs=10 , 
    validation_split = 0.2,
    validation_data = (X_test , y_test)
)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6778 - loss: 0.5789 - val_accuracy: 0.6753 - val_loss: 0.5395
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7616 - loss: 0.5017 - val_accuracy: 0.6970 - val_loss: 0.5014
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7765 - loss: 0.4727 - val_accuracy: 0.7446 - val_loss: 0.4743
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7952 - loss: 0.4492 - val_accuracy: 0.7835 - val_loss: 0.4527
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8138 - loss: 0.4280 - val_accuracy: 0.8009 - val_loss: 0.4378
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8175 - loss: 0.4115 - val_accuracy: 0.8182 - val_loss: 0.4198
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8343 - loss: 0.3976 - val_accuracy: 0.8225 - val_loss: 0.4085
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8399 - loss: 0.3852 - val_accuracy: 0.8442 - val_loss:

In [72]:
y_pred = model.predict(X_test)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


In [73]:
from sklearn.metrics import accuracy_score
y_pred_binary = (y_pred > 0.5).astype(int)
print(f"{accuracy_score(y_test , y_pred_binary):.3f}")

0.844


In [74]:
import keras_tuner as kt

def BuildModel_1(hp):

    model = Sequential([
        layers.Dense(
            5,
            activation="relu",
            input_shape=(X_train.shape[1],)
        ),

        layers.Dense(
            3,
            activation="relu"
        ),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=hp.Choice(
            "optimizer",
            values=["adam", "sgd", "rmsprop", "adadelta"]
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [75]:
tuner1 = kt.RandomSearch(
    BuildModel_1,
    objective="val_accuracy",
    max_trials=4,
    directory="myDir",
    project_name="tuner1",
    overwrite=True
)

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [76]:
tuner1.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test,y_test)
)

Trial 4 Complete [00h 00m 01s]
val_accuracy: 0.6709956526756287

Best val_accuracy So Far: 0.6883116960525513
Total elapsed time: 00h 00m 05s


In [77]:
best_hp1 = tuner1.get_best_hyperparameters(num_trials=1)[0]

print(best_hp1.values)

{'optimizer': 'adam'}


In [78]:
best_model1 = tuner1.get_best_models(num_models=1)[0]

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [79]:
history1 = best_model1.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=100,
    initial_epoch=5,
    validation_data=(X_test, y_test)
)

Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6555 - loss: 0.6754 - val_accuracy: 0.6883 - val_loss: 0.6663
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6574 - loss: 0.6687 - val_accuracy: 0.6797 - val_loss: 0.6579
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6685 - loss: 0.6605 - val_accuracy: 0.6840 - val_loss: 0.6484
Epoch 9/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6723 - loss: 0.6514 - val_accuracy: 0.6840 - val_loss: 0.6386
Epoch 10/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6741 - loss: 0.6425 - val_accuracy: 0.6883 - val_loss: 0.6280
Epoch 11/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6760 - loss: 0.6323 - val_accuracy: 0.6926 - val_loss: 0.6180
Epoch 12/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6778 - loss: 0.6227 - val_accuracy: 0.6970 - val_loss: 0.6077
Epoch 13/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6778 - loss: 0.6128 - val_accuracy: 0.6926

In [80]:
def buildModel_2(hp):

    model = Sequential([
        layers.Dense(
            units=hp.Int(
                "units",
                min_value=8,
                max_value=128
            ),
            activation="relu",
            input_shape=(X_train.shape[1],)
        ),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=keras.optimizers.RMSprop(
            learning_rate=0.01
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [81]:
tuner2 = kt.RandomSearch(
    buildModel_2,
    objective="val_loss",
    max_trials=5,
    directory="myDir",
    project_name="tuner2",
    overwrite=True
)

In [82]:
tuner2.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

Trial 5 Complete [00h 00m 01s]
val_loss: 0.3797158896923065

Best val_loss So Far: 0.37647905945777893
Total elapsed time: 00h 00m 06s


In [87]:
best_hp2 = tuner2.get_best_hyperparameters(num_trials=1)[0]
print(best_hp2.values)

{'units': 125}


In [88]:
best_model2 = tuner2.get_best_models(num_models=1)[0]